In [1]:
import os
import json
import pandas as pd
from bert_score import score
from src import NewsAnalyzerAPI as na

d:\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
con = na.LLMConnector("http://localhost:1234/v1")
analyzer = na.NewsAnalyzer(con)

In [3]:
def extract_texts(file_path, data_list):
    def extract_component(component):
        annotations = data['fiveWoneH'][component]['annotated']
        texts = [item.get('text') for item in annotations]
        return "; ".join(text for text in texts if text is not None)

    with open(file_path, 'r') as file:
        data = json.load(file)

    text = data['text']
    what_true = extract_component('what')
    where_true = extract_component('where')
    when_true = extract_component('when')
    who_true = extract_component('who')
    why_true = extract_component('why')
    how_true = extract_component('how')

    article = na.NewsArticle("", "", data['text'], "", "")
    what_pred = analyzer.identify_what(article)
    where_pred = analyzer.identify_where(article)
    when_pred = analyzer.identify_when(article)
    who_pred = analyzer.identify_who(article)
    why_pred = analyzer.identify_why(article)
    how_pred = analyzer.identify_how(article)

    data_list.append({
        'text': text,
        'what_true': what_true,
        'where_true': where_true,
        'when_true': when_true,
        'who_true': who_true,
        'why_true': why_true,
        'how_true': how_true,
        'what_pred': what_pred,
        'where_pred': where_pred,
        'when_pred': when_pred,
        'who_pred': who_pred,
        'why_pred': why_pred,
        'how_pred': how_pred
    })

def evaluate(data_list):
    for article in data_list:
        cands = [article[component] for component in article if component.endswith('_pred')]
        refs = [article[component] for component in article if component.endswith('_true')]
        P, R, F1 = score(cands, refs, lang="en")
        print(F1)
        print(f"System level F1 score: {F1.mean():.3f}")

In [4]:
import logging
from transformers import logging as transformers_logging

# Configuring log level to suppress unwanted warnings
logging.basicConfig(level=logging.ERROR)
transformers_logging.set_verbosity_error()

In [5]:
data_list = []
extract_texts('./data_samples/0e5fa7c0e6252bfeeea5e3840c6cb503f299c19d24331c4ba60c5974.json', data_list)
print(data_list)
evaluate(data_list)

[{'text': 'Skip Ad Ad Loading... x Embed x Share Toblerone is facing a mountain of criticism for changing the shape of its famous triangular candy bars in British stores, a move it blames on rising costs. USA TODAY Toblerone chocolate bars come in a variety of sizes, but recently changed the shape of two of its smaller bars sold in the UK. (Photo: Martin Ruetschi, AP) The UK has a chocolate bar crisis on its hands: the beloved Swiss chocolate bar is unrecognizable. Toblerone, the classic chocolate bar with almond-and-honey-filled triangle chunks, recently lost weight. In two sizes, the triangles shrunk, leaving wider gaps of chocolate. Toblerone can you tell me what this is all about... looks like there\'s half a bar missing! pic.twitter.com/C2VD3DjppE -- Alana Cartwright (@AlanaCartwrigh3) October 29, 2016  @HelenRyles Hi Helen, yes this is just our smaller bar. -- Toblerone (@Toblerone) October 31, 2016  The 400-gram bar was reduced to a 360-gram bar and the 170-gram was reduced to 1

In [ ]:
# Extracting components for 10 articles and writing in a spreadsheet
# data_list = []
# data_folder = './data_samples/'
# cnt = 0
# for filename in os.listdir(data_folder):
#     cnt += 1
#     if cnt > 10:
#         break
    
#    file_path = os.path.join(data_folder, filename)
#    extract_texts(file_path, data_list)
# df = pd.DataFrame(data_list)
# df.to_excel('news.xlsx', index=False)
# df.to_csv('news.csv', index=False, encoding='utf-8')

In [6]:
df = pd.read_excel('news.xlsx')
df = df.fillna('')
data_list2 = df.to_dict(orient='records')

In [7]:
evaluate(data_list2)

tensor([0.8409, 0.8634, 0.8200, 0.8056, 0.0000, 0.0000])
System level F1 score: 0.555


tensor([0.8377, 0.8273, 0.8091, 0.8773, 0.7772, 0.0000])
System level F1 score: 0.688


tensor([0.8144, 0.7973, 0.8129, 0.7755, 0.0000, 0.8111])
System level F1 score: 0.669
tensor([0.8418, 0.8209, 0.8490, 0.8324, 0.8510, 0.8005])
System level F1 score: 0.833
tensor([0.8255, 0.8987, 0.7883, 0.7957, 0.8149, 0.8031])
System level F1 score: 0.821
tensor([0.8506, 0.8699, 0.8057, 0.8486, 0.8503, 0.7953])
System level F1 score: 0.837
tensor([0.8020, 0.8058, 0.8312, 0.8162, 0.8417, 0.8001])
System level F1 score: 0.816


tensor([0.8014, 0.8385, 0.8392, 0.7650, 0.8320, 0.0000])
System level F1 score: 0.679
tensor([0.7996, 0.8694, 0.7995, 0.8295, 0.8302, 0.8290])
System level F1 score: 0.826
tensor([0.7975, 0.8812, 0.8040, 0.7976, 0.8759, 0.8312])
System level F1 score: 0.831
